# READ DATA

In [100]:
import numpy as np
import pandas as pd 
import seaborn as sns 

factor = pd.read_csv("../factors.csv")
report = pd.read_csv("../report.csv")

In [101]:
product_num = len(factor["product_id"].unique())
order_num = len(factor["order_id"].unique())
products = factor.drop_duplicates(subset=["product_id"])
departments = factor.drop_duplicates(subset=["department_id"])
prd_dept = factor.drop_duplicates(subset=["product_id"])

print("productNum:" , product_num)
print("orderNum:",order_num)

productNum: 39123
orderNum: 131209


In [102]:
products = products[['product_id','product_name']]
products

,product_id,product_name
0,49302,Bulgarian Yogurt
1,11109,Organic 4% Milk Fat Whole Milk Cottage Cheese
2,10246,Organic Celery Hearts
3,49683,Cucumber Kirby
4,43633,Lightly Smoked Sardines in Olive Oil
...,...,...
1384121,1528,Sprinkles Decors
1384146,47935,Classic Original Lip Balm SPF 12
1384147,9491,Goats Milk & Chai Soap
1384160,16380,Stevia Sweetener


In [103]:
departments = departments[['department_id','department_name']]
departments

,department_id,department_name
0,16,dairy eggs
2,4,produce
4,15,canned goods
9,7,beverages
13,20,deli
16,19,snacks
18,13,pantry
21,1,frozen
41,12,meat seafood
49,17,household


In [104]:
prd_dept = prd_dept[['product_id','department_id','department_name']]
prd_dept

,product_id,department_id,department_name
0,49302,16,dairy eggs
1,11109,16,dairy eggs
2,10246,4,produce
3,49683,4,produce
4,43633,15,canned goods
...,...,...,...
1384121,1528,13,pantry
1384146,47935,11,personal care
1384147,9491,11,personal care
1384160,16380,13,pantry


In [105]:
factor.count()
factor.head()

,order_id,product_id,product_name,department_id,department_name
0,1,49302,Bulgarian Yogurt,16,dairy eggs
1,1,11109,Organic 4% Milk Fat Whole Milk Cottage Cheese,16,dairy eggs
2,1,10246,Organic Celery Hearts,4,produce
3,1,49683,Cucumber Kirby,4,produce
4,1,43633,Lightly Smoked Sardines in Olive Oil,15,canned goods


# PART A.1

In [106]:
factor1 = factor[['order_id','product_id']]
joined = pd.merge(factor1,factor1, on="order_id", how="inner")


In [107]:
solid = joined.drop(['order_id'],axis=1)
counted = solid.groupby(['product_id_x', 'product_id_y']).size().reset_index(name='count')
df = counted[counted['product_id_x'] != counted['product_id_y']]

In [108]:
df['support'] = df['count']/order_num
df = df[df['support'] > 0.003]


C:\Users\Ata\AppData\Local\Temp\ipykernel_2076\659519033.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['support'] = df['count']/order_num


In [109]:
df

,product_id_x,product_id_y,count,support
476298,2295,15290,396,0.003018
963534,4605,21903,544,0.004146
964134,4605,24852,1071,0.008163
964419,4605,26209,496,0.003780
968998,4605,47626,747,0.005693
...,...,...,...,...
11356281,49235,13176,490,0.003734
11358365,49235,24852,500,0.003811
11458461,49683,24852,743,0.005663
11462224,49683,47626,424,0.003231


In [110]:
each_product = factor.drop(['order_id'],axis=1).groupby(['product_id']).size().reset_index(name='count')
df['confidence'] = df['count'] / df['product_id_x'].map(each_product.set_index('product_id')['count'])
df = df[df['confidence'] > 0.25]
df.reset_index()


,index,product_id_x,product_id_y,count,support,confidence
0,476298,2295,15290,396,0.003018,0.339623
1,964134,4605,24852,1071,0.008163,0.284689
2,1034677,4920,24852,1162,0.008856,0.286277
3,1097777,5077,24852,628,0.004786,0.273281
4,1198680,5450,24852,865,0.006593,0.278762
5,1324176,5876,13176,1067,0.008132,0.304422
6,1825860,8174,13176,725,0.005526,0.366162
7,1862237,8277,13176,687,0.005236,0.305062
8,1893548,8424,24852,925,0.007050,0.315484
9,2045824,9076,24852,716,0.005457,0.308222


In [111]:
df1 = df
df1 = pd.merge(df1, products , left_on="product_id_x" , right_on= "product_id" , how= 'inner')
df1.drop(['product_id'],axis = 1,inplace=True)
df1 = pd.merge(df1, products , left_on="product_id_y" , right_on= "product_id" , how= 'inner')
df1.drop(['product_id','count'],axis = 1,inplace=True)
df1 = df1.rename(columns={'product_id_x': 'product_id_1', 'product_id_y' : 'product_id_2' ,'product_name_x' : 'product_name_1','product_name_y':'product_name_2'})
df1 = df1[['product_id_1','product_id_2','product_name_1','product_name_2','confidence','support']]
df1

,product_id_1,product_id_2,product_name_1,product_name_2,confidence,support
0,2295,15290,Yellow Bell Pepper,Orange Bell Pepper,0.339623,0.003018
1,4605,24852,Yellow Onions,Banana,0.284689,0.008163
2,4920,24852,Seedless Red Grapes,Banana,0.286277,0.008856
3,5077,24852,100% Whole Wheat Bread,Banana,0.273281,0.004786
4,5450,24852,Small Hass Avocado,Banana,0.278762,0.006593
5,5876,13176,Organic Lemon,Bag of Organic Bananas,0.304422,0.008132
6,8174,13176,Organic Navel Orange,Bag of Organic Bananas,0.366162,0.005526
7,8277,13176,Apple Honeycrisp Organic,Bag of Organic Bananas,0.305062,0.005236
8,8424,24852,Broccoli Crown,Banana,0.315484,0.007050
9,9076,24852,Blueberries,Banana,0.308222,0.005457


In [112]:
df1.to_csv('A1.csv')

# PART A.2

In [113]:
factor.head()

,order_id,product_id,product_name,department_id,department_name
0,1,49302,Bulgarian Yogurt,16,dairy eggs
1,1,11109,Organic 4% Milk Fat Whole Milk Cottage Cheese,16,dairy eggs
2,1,10246,Organic Celery Hearts,4,produce
3,1,49683,Cucumber Kirby,4,produce
4,1,43633,Lightly Smoked Sardines in Olive Oil,15,canned goods


In [114]:
factor1 = factor[['order_id','department_id']]
joined = pd.merge(factor1,factor1, on="order_id", how="inner")
joined


,order_id,department_id_x,department_id_y
0,1,16,16
1,1,16,16
2,1,16,4
3,1,16,4
4,1,16,15
...,...,...,...
22868452,3421070,13,13
22868453,3421070,13,4
22868454,3421070,4,16
22868455,3421070,4,13


In [115]:
unique_pair = joined.groupby(['order_id','department_id_x', 'department_id_y']).size().reset_index(name='count')
solid= unique_pair.drop(['order_id','count'],axis=1)
df = solid.groupby(['department_id_x', 'department_id_y']).size().reset_index(name='count')
df

,department_id_x,department_id_y,count
0,1,1,51071
1,1,2,826
2,1,3,18560
3,1,4,41264
4,1,5,1038
...,...,...,...
436,21,17,1177
437,21,18,612
438,21,19,3878
439,21,20,2373


In [116]:
df['support'] = df['count']/order_num
df = df[df['support'] > 0.003]


In [117]:
df

,department_id_x,department_id_y,count,support
0,1,1,51071,0.389234
1,1,2,826,0.006295
2,1,3,18560,0.141454
3,1,4,41264,0.314491
4,1,5,1038,0.007911
...,...,...,...,...
436,21,17,1177,0.008970
437,21,18,612,0.004664
438,21,19,3878,0.029556
439,21,20,2373,0.018086


In [118]:
each_department = factor.drop(['order_id'],axis=1).groupby(['department_id']).size().reset_index(name='count')
df['confidence'] = df['count'] / df['department_id_x'].map(each_department.set_index('department_id')['count'])
df = df[df['confidence'] > 0.25]
df.reset_index()


C:\Users\Ata\AppData\Local\Temp\ipykernel_2076\388991405.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['confidence'] = df['count'] / df['department_id_x'].map(each_department.set_index('department_id')['count'])


,index,department_id_x,department_id_y,count,support,confidence
0,0,1,1,51071,0.389234,0.508544
1,3,1,4,41264,0.314491,0.410890
2,6,1,7,26707,0.203545,0.265937
3,15,1,16,39403,0.300307,0.392359
4,18,1,19,27458,0.209269,0.273415
...,...,...,...,...,...,...
136,434,21,15,2106,0.016051,0.255242
137,435,21,16,5498,0.041903,0.666343
138,438,21,19,3878,0.029556,0.470004
139,439,21,20,2373,0.018086,0.287602


In [119]:
df2 = df
df2 = pd.merge(df2, departments , left_on="department_id_x" , right_on= "department_id" , how= 'inner')
df2.drop(['department_id'],axis = 1,inplace=True)
df2 = pd.merge(df2, departments , left_on="department_id_y" , right_on= "department_id" , how= 'inner')
df2.drop(['department_id','count'],axis = 1,inplace=True)
df2 = df2.rename(columns={'department_id_x': 'department_id_1', 'department_id_y' : 'department_id_2' ,'department_name_x' : 'department_name_1','department_name_y':'department_name_2'})
df2 = df2[['department_id_1','department_id_2','department_name_1','department_name_2','confidence','support']]
df2

,department_id_1,department_id_2,department_name_1,department_name_2,confidence,support
0,1,1,frozen,frozen,0.508544,0.389234
1,1,4,frozen,produce,0.410890,0.314491
2,1,7,frozen,beverages,0.265937,0.203545
3,1,16,frozen,dairy eggs,0.392359,0.300307
4,1,19,frozen,snacks,0.273415,0.209269
...,...,...,...,...,...,...
136,21,15,missing,canned goods,0.255242,0.016051
137,21,16,missing,dairy eggs,0.666343,0.041903
138,21,19,missing,snacks,0.470004,0.029556
139,21,20,missing,deli,0.287602,0.018086


In [120]:
df2.to_csv('A2.csv')

# PART B


In [121]:
df1
df3 = df1
df3 = pd.merge(df3, prd_dept , left_on="product_id_1" , right_on= "product_id" , how= 'inner')
df3.drop(['product_id'],axis = 1,inplace=True)
df3 = pd.merge(df3, prd_dept , left_on="product_id_2" , right_on= "product_id" , how= 'inner')
df3.drop(['product_id'],axis = 1,inplace=True)
df3 = df3[df3['department_id_x'] == df3['department_id_y']].reset_index()
df3.drop(['index','department_id_y','department_name_y'],axis=1,inplace=True)
df3.rename(columns={'department_id_x':'depatment_id','department_name_x' : 'department_name' }, inplace=True)
df3 = df3[['depatment_id','department_name','product_id_1','product_id_2','product_name_1','product_name_2','confidence','support']]
df3


,depatment_id,department_name,product_id_1,product_id_2,product_name_1,product_name_2,confidence,support
0,4,produce,2295,15290,Yellow Bell Pepper,Orange Bell Pepper,0.339623,0.003018
1,4,produce,4605,24852,Yellow Onions,Banana,0.284689,0.008163
2,4,produce,4920,24852,Seedless Red Grapes,Banana,0.286277,0.008856
3,4,produce,5450,24852,Small Hass Avocado,Banana,0.278762,0.006593
4,4,produce,5876,13176,Organic Lemon,Bag of Organic Bananas,0.304422,0.008132
5,4,produce,8174,13176,Organic Navel Orange,Bag of Organic Bananas,0.366162,0.005526
6,4,produce,8277,13176,Apple Honeycrisp Organic,Bag of Organic Bananas,0.305062,0.005236
7,4,produce,8424,24852,Broccoli Crown,Banana,0.315484,0.007050
8,4,produce,9387,24852,Granny Smith Apples,Banana,0.314721,0.003308
9,4,produce,9839,13176,Organic Broccoli,Bag of Organic Bananas,0.317604,0.004001


In [122]:
df3.to_csv('B.csv')